# Huấn luyện phối đồ từ danh mục Polyvore
Chọn runtime GPU và chạy các ô theo thứ tự để huấn luyện embedding. Nếu chuyển lượt đang chạy, dùng checkpoint đã lưu và cùng thư mục Drive như hướng dẫn.

In [ ]:
MODEL = "embedding"  # Polyvore catalog
REQUIRE_GPU = True  # @param {type:"boolean"}
import sys, os, json, subprocess
from pathlib import Path
import torch, torchvision
assert MODEL == "embedding"
if REQUIRE_GPU and not torch.cuda.is_available():
    raise RuntimeError("Phiên này chưa có GPU. Chọn runtime GPU hoặc bỏ REQUIRE_GPU nếu muốn chạy CPU.")
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
root = Path("/content/phoi_do")
(root / "scripts").mkdir(parents=True, exist_ok=True)
def run(*args):
    subprocess.run([sys.executable, "-u", *map(str, args)], cwd=root, check=True)
run("-m", "pip", "install", "-q", "pyarrow", "huggingface_hub", "requests", "pillow")

### Mã nguồn dùng để chạy
Ô sau ghi các script vào thư mục tạm `/content/phoi_do/scripts`. Có thể mở các tệp trong bảng Files của Colab để đọc trước khi chạy phần huấn luyện.

In [ ]:
sources = {'check_training_status.py': '"""Report completion from actual artifacts; pilot outputs never count as full training."""\nimport json\nfrom pathlib import Path\n\nROOT = Path(__file__).absolute().parents[1]\n\n\ndef main():\n    checks = {}\n    for dataset in (\'polyvore\',):\n        path = ROOT / f\'data/processed/{dataset}/preprocessing_report.json\'\n        checks[dataset + \'_preprocessed\'] = path.is_file()\n    for model in (\'embedding\',):\n        path = ROOT / f\'artifacts/{model}/full/training_report.json\'\n        complete = False\n        if path.exists():\n            report = json.loads(path.read_text())\n            complete = not report.get(\'pilot\', True) and bool(report.get(\'test\')) and bool(report.get(\'epochs\'))\n        checks[model + \'_full_training\'] = complete\n    checks[\'association_rules\'] = (ROOT / \'artifacts/mining/fpgrowth_rules.json\').is_file()\n    checks[\'image_compatibility_evaluated\'] = bool(list((ROOT/\'reports/metrics\').glob(\'compatibility_image_*.json\')))\n    checks[\'image_ranking_evaluated\'] = bool(list((ROOT/\'reports/metrics\').glob(\'ranking_image_*.json\')))\n    print(json.dumps({\'checks\': checks, \'ready_for_result_review\': all(checks.values()),\n                      \'note\': \'Review quality and limitations before creating UI; artifact existence is not an accuracy guarantee.\'}, indent=2))\n\n\nif __name__ == \'__main__\': main()\n', 'download_datasets.py': '"""Download official datasets, resume partial HTTP files and validate assets.\n\nCredentials are read by huggingface_hub from the existing local login only.\n"""\nimport argparse\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\n\nos.environ.setdefault(\'HF_HUB_DISABLE_XET\', \'1\')\nROOT = Path(__file__).absolute().parents[1]\n\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with path.open(\'rb\') as f:\n        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b\'\'):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef polyvore(reuse):\n    from huggingface_hub import HfApi, hf_hub_download\n    repo = \'mvasil/polyvore-outfits\'\n    info = HfApi().dataset_info(repo, files_metadata=True)\n    base = ROOT / \'data/raw/polyvore\'\n    wanted = [s for s in info.siblings if s.rfilename.startswith((\'disjoint/\', \'data/disjoint/\'))\n              or s.rfilename in (\'categories.csv\', \'polyvore_item_metadata.json\', \'README.md\')]\n    records = []\n    for s in wanted:\n        target = base / s.rfilename\n        target.parent.mkdir(parents=True, exist_ok=True)\n        old = Path(reuse) / s.rfilename if reuse else None\n        if not target.exists() and old and old.is_file() and old.stat().st_size == s.size:\n            shutil.copyfile(old, target)\n            print(f\'Reused {s.rfilename}\', flush=True)\n        if not target.exists() or target.stat().st_size != s.size:\n            hf_hub_download(repo, s.rfilename, repo_type=\'dataset\', revision=info.sha, local_dir=base)\n        if target.stat().st_size != s.size:\n            raise ValueError(f\'Invalid size: {s.rfilename}\')\n        digest = sha256(target)\n        expected_hash = getattr(s.lfs, \'sha256\', None) if s.lfs else None\n        if expected_hash and digest != expected_hash:\n            raise ValueError(f\'Checksum mismatch: {s.rfilename}\')\n        records.append({\'path\': s.rfilename, \'size\': s.size, \'sha256\': digest})\n        print(f\'Validated {s.rfilename}\', flush=True)\n    (base / \'manifest.json\').write_text(json.dumps({\'revision\': info.sha, \'files\': records}, indent=2), encoding=\'utf-8\')\n\n\nif __name__ == \'__main__\':\n    p = argparse.ArgumentParser()\n    p.add_argument(\'--dataset\', choices=[\'polyvore\'], required=True)\n    p.add_argument(\'--reuse\')\n    args = p.parse_args()\n    polyvore(args.reuse)\n', 'evaluate_recommendation.py': '"""Rank complete triples; evaluate controlled candidate pools, not human taste.\n\nAll queries are held-out positive triples, with the original bottom/shoes plus\nsame-split distractors. Candidate construction is identical across variants.\n"""\nimport argparse\nimport hashlib\nimport itertools\nimport json\nfrom pathlib import Path\nimport random\nimport joblib\nimport numpy as np\n\nROOT = Path(__file__).absolute().parents[1]\n\n\ndef feature_row(ids, items):\n    a,b,c = [items[i][\'category\'] for i in ids]\n    return {\'top\':a, \'bottom\':b, \'shoes\':c, \'top_bottom\':a+\'|\'+b, \'top_shoes\':a+\'|\'+c, \'bottom_shoes\':b+\'|\'+c}\n\n\ndef mmr(candidates, relevance, k, lam):\n    selected, remaining = [], list(range(len(candidates)))\n    while remaining and len(selected) < k:\n        def objective(i):\n            overlap = max((sum(x == y for x,y in zip(candidates[i][1:], candidates[j][1:]))/2 for j in selected), default=0)\n            return lam*relevance[i] - (1-lam)*overlap\n        best = max(remaining, key=objective)\n        selected.append(best); remaining.remove(best)\n    return selected\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument(\'--model-dir\', type=Path)\n    p.add_argument(\'--embeddings\', type=Path)\n    p.add_argument(\'--queries\', type=int, default=100)\n    p.add_argument(\'--per-slot\', type=int, default=10)\n    args = p.parse_args()\n    data = ROOT / \'data/processed/polyvore\'\n    items = json.loads((data / \'items.json\').read_text())\n    expected_version = (\'image_\' + hashlib.sha256(args.embeddings.read_bytes()).hexdigest()[:12]) if args.embeddings else \'metadata_baseline\'\n    model_dir = args.model_dir or ROOT / \'artifacts/compatibility\' / expected_version\n    bundle = joblib.load(model_dir / \'random_forest.joblib\')\n    if bundle[\'feature_version\'] != expected_version:\n        raise ValueError(\'Embedding version does not match compatibility model\')\n    embeddings = None\n    if args.embeddings:\n        a = np.load(args.embeddings, allow_pickle=False)\n        embeddings = dict(zip(a[\'ids\'].astype(str), a[\'vectors\']))\n    if bundle[\'feature_version\'].startswith(\'image_\') and embeddings is None:\n        raise ValueError(\'Image model requires its embeddings\')\n    rules = json.loads((ROOT / \'artifacts/mining/fpgrowth_rules.json\').read_text())\n    prepared = {}\n    for split in (\'validation\', \'test\'):\n        rows = [r for r in json.loads((data / f\'{split}_examples.json\').read_text()) if r[\'label\'] == 1]\n        rng = random.Random(71); rng.shuffle(rows)\n        # Keep candidate pool within exported embeddings when applicable.\n        pools = {s:sorted(i for i,v in items.items() if v[\'slot\']==s and v[\'split\']==split and (embeddings is None or i in embeddings)) for s in (\'bottom\',\'shoes\')}\n        prepared[split] = []\n        for row in rows[:args.queries]:\n            truth = row[\'item_ids\']\n            options = []\n            for slot, iid in zip((\'bottom\',\'shoes\'), truth[1:]):\n                others = [i for i in pools[slot] if i != iid]\n                choices = [iid] + rng.sample(others, min(args.per_slot-1, len(others)))\n                rng.shuffle(choices); options.append(choices)\n            candidates = [(truth[0],b,s) for b,s in itertools.product(*options)]\n            X = bundle[\'vectorizer\'].transform([feature_row(c,items) for c in candidates])\n            if embeddings is not None:\n                visual=[]\n                for c in candidates:\n                    a,b,d = [embeddings[i] for i in c]\n                    visual.append(np.concatenate([a,b,d,a*b,a*d,b*d]))\n                X=np.hstack([X,visual])\n            compatibility = bundle[\'model\'].predict_proba(X)[:,1]\n            rule_scores=[]\n            for c in candidates:\n                tokens = {items[i][\'slot\']+\':\'+items[i][\'category\'] for i in c}\n                rule_scores.append(max((r[\'confidence\'] for r in rules if set(r[\'antecedents\'])|set(r[\'consequents\']) <= tokens),default=0))\n            prepared[split].append((candidates, tuple(truth), compatibility, np.array(rule_scores)))\n    def assess(split, alpha, lam):\n        hits, reciprocal, diversity = [],[],[]\n        for candidates,truth,comp,rule in prepared[split]:\n            selected = mmr(candidates, alpha*comp+(1-alpha)*rule, 5, lam)\n            outfits=[candidates[i] for i in selected]\n            hits.append(truth in outfits)\n            reciprocal.append(1/(outfits.index(truth)+1) if truth in outfits else 0)\n            pair_sim=[sum(x==y for x,y in zip(a[1:],b[1:]))/2 for a,b in itertools.combinations(outfits,2)]\n            diversity.append(1-np.mean(pair_sim) if pair_sim else 0)\n        return {\'hit_at_5\':float(np.mean(hits)), \'mrr_at_5\':float(np.mean(reciprocal)),\n                \'diversity_at_5\':float(np.mean(diversity)), \'queries\':len(hits)}\n    experiments=[]\n    for a in (1.0,0.75,0.5):\n        for lam in (1.0,0.9,0.7):\n            m=assess(\'validation\',a,lam)\n            experiments.append({\'alpha\':a,\'lambda\':lam,**m})\n    best=max(experiments,key=lambda r:r[\'hit_at_5\']+0.05*r[\'diversity_at_5\'])\n    report={\'protocol\':\'original full triple retrieval among same-split controlled candidates; original guaranteed present\',\n            \'candidates_per_query\':args.per_slot**2,\'feature_version\':bundle[\'feature_version\'],\n            \'validation_selection_objective\':\'Hit@5 + 0.05*diversity@5\', \'validation_search\':experiments,\n            \'chosen\':best,\'test_baseline\':assess(\'test\',1,1), \'test_selected\':assess(\'test\',best[\'alpha\'],best[\'lambda\'])}\n    out=ROOT/\'reports/metrics\';out.mkdir(parents=True,exist_ok=True)\n    (out/f"ranking_{bundle[\'feature_version\']}.json").write_text(json.dumps(report,indent=2))\n    print(json.dumps(report,indent=2))\n\n\nif __name__==\'__main__\': main()\n', 'export_embeddings.py': '"""Export vectors tied to a trained checkpoint; reusable by compatibility training."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport numpy as np\nfrom PIL import Image\nimport torch\nfrom train_embedding import ROOT, Encoder, transform\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument(\'--checkpoint\', type=Path, required=True)\n    p.add_argument(\'--batch-size\', type=int, default=16)\n    p.add_argument(\'--output\', type=Path, default=ROOT / \'artifacts/embedding/catalog_vectors.npz\')\n    args = p.parse_args()\n    torch.set_num_threads(4)\n    device = \'cuda\' if torch.cuda.is_available() else \'cpu\'\n    model = Encoder(pretrained=False).to(device)\n    model.load_state_dict(torch.load(args.checkpoint, map_location=device, weights_only=False)[\'model\'])\n    model.eval()\n    items = json.loads((ROOT / \'data/processed/polyvore/items.json\').read_text())\n    # All items used by examples, across the official item-disjoint splits.\n    required = set()\n    for split in (\'train\', \'validation\', \'test\'):\n        rows = json.loads((ROOT / f\'data/processed/polyvore/{split}_examples.json\').read_text())\n        required.update(i for r in rows for i in r[\'item_ids\'])\n    ids = sorted(required)\n    vectors = []\n    tfm = transform()\n    with torch.no_grad():\n        for start in range(0, len(ids), args.batch_size):\n            batch = []\n            for iid in ids[start:start+args.batch_size]:\n                with Image.open(ROOT / items[iid][\'image_path\']) as image:\n                    batch.append(tfm(image.convert(\'RGB\')))\n            vectors.append(model(torch.stack(batch).to(device)).cpu().numpy())\n            if start % (args.batch_size*50) == 0: print(f\'Encoded {start+len(batch)}/{len(ids)}\', flush=True)\n    args.output.parent.mkdir(parents=True, exist_ok=True)\n    np.savez_compressed(args.output, ids=np.array(ids), vectors=np.concatenate(vectors))\n    args.output.with_suffix(\'.json\').write_text(json.dumps({\'checkpoint\': str(args.checkpoint),\n        \'checkpoint_sha256\': hashlib.sha256(args.checkpoint.read_bytes()).hexdigest(), \'count\': len(ids)}, indent=2))\n\n\nif __name__ == \'__main__\': main()\n', 'preprocess_polyvore.py': '"""Standalone preprocessing. Preserve official item-disjoint splits.\n\nThree-slot examples use positive outfit projections and explicitly synthetic\nsame-slot replacements. They are NOT the official compatibility benchmark.\n"""\nimport argparse\nimport csv\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport random\nimport itertools\nfrom PIL import Image\nimport pyarrow.parquet as pq\n\nROOT = Path(__file__).absolute().parents[1]\nSLOTS = (\'top\', \'bottom\', \'shoes\')\nMAP = {\'tops\': \'top\', \'bottoms\': \'bottom\', \'shoes\': \'shoes\'}\n\n\ndef dump(path, data):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n\n\ndef main():\n    raw = ROOT / \'data/raw/polyvore\'\n    out = ROOT / \'data/processed/polyvore\'\n    out.mkdir(parents=True, exist_ok=True)\n    metadata = json.loads((raw / \'polyvore_item_metadata.json\').read_text(encoding=\'utf-8\'))\n    categories = {}\n    with (raw / \'categories.csv\').open(encoding=\'utf-8\') as f:\n        for row in csv.reader(f):\n            if len(row) >= 2:\n                categories.setdefault(row[0], row[1])\n    items, examples, transactions, report, seen = {}, {}, {}, {}, {}\n    image_dir = out / \'images\'\n    image_dir.mkdir(exist_ok=True)\n    reserved_ids = set()\n    for split, original in [(\'train\', \'train\'), (\'validation\', \'valid\'), (\'test\', \'test\')]:\n        original_outfits = json.loads((raw / \'disjoint\' / f\'{original}.json\').read_text())\n        source_ids = {str(x[\'item_id\']) for o in original_outfits for x in o[\'items\']}\n        overlap = source_ids & reserved_ids\n        outfits = [o for o in original_outfits if not any(str(x[\'item_id\']) in reserved_ids for x in o[\'items\'])]\n        reserved_ids.update(source_ids)\n        wanted = {str(x[\'item_id\']) for o in outfits for x in o[\'items\']}\n        seen[split] = wanted\n        count = invalid = 0\n        for batch in pq.ParquetFile(raw / \'data/disjoint\' / f\'{split}.parquet\').iter_batches(batch_size=64):\n            for row in batch.to_pylist():\n                iid = str(row[\'item_id\'])\n                m = metadata.get(iid, {})\n                slot = MAP.get(m.get(\'semantic_category\'))\n                if iid not in wanted or slot is None:\n                    continue\n                path = image_dir / f\'{iid}.jpg\'\n                try:\n                    if not path.exists():\n                        im = Image.open(io.BytesIO(row[\'image\'][\'bytes\'])).convert(\'RGB\')\n                        im.save(path, quality=95)\n                    with Image.open(path) as im:\n                        im.verify()\n                except Exception:\n                    invalid += 1\n                    continue\n                items[iid] = {\'item_id\': iid, \'slot\': slot, \'split\': split,\n                              \'category\': categories.get(str(m.get(\'category_id\')), \'unknown\'),\n                              \'description\': m.get(\'description\') or m.get(\'title\') or m.get(\'url_name\', \'\'),\n                              \'image_path\': path.relative_to(ROOT).as_posix()}\n                count += 1\n        positive, seen_triples = [], set()\n        for o in outfits:\n            groups = {s: [] for s in SLOTS}\n            for x in o[\'items\']:\n                iid = str(x[\'item_id\'])\n                if iid in items:\n                    groups[items[iid][\'slot\']].append(iid)\n            if not all(groups.values()):\n                continue\n            # One deterministic triple per outfit; avoids multiplying near duplicates.\n            ids = tuple(sorted(set(groups[s]))[0] for s in SLOTS)\n            if ids not in seen_triples:\n                seen_triples.add(ids)\n                positive.append({\'item_ids\': list(ids), \'label\': 1,\n                                 \'source_outfit_id\': str(o[\'set_id\']),\n                                 \'label_source\': \'projection_of_curated_outfit\'})\n        pools = {s: sorted(i for i in wanted if i in items and items[i][\'slot\'] == s) for s in SLOTS}\n        rng = random.Random(42)\n        negatives, used_negative = [], set()\n        for row in positive:\n            for _ in range(50):\n                ids = list(row[\'item_ids\'])\n                slot_idx = rng.randrange(3)\n                ids[slot_idx] = rng.choice(pools[SLOTS[slot_idx]])\n                key = tuple(ids)\n                if key not in seen_triples and key not in used_negative:\n                    negatives.append({\'item_ids\': ids, \'label\': 0,\n                                      \'source_outfit_id\': row[\'source_outfit_id\'],\n                                      \'label_source\': \'synthetic_same_slot_replacement\'})\n                    used_negative.add(key)\n                    break\n        examples[split] = positive + negatives\n        transactions[split] = [[items[i][\'slot\'] + \':\' + items[i][\'category\'] for i in r[\'item_ids\']] for r in positive]\n        report[split] = {\'original_outfits\': len(original_outfits), \'retained_outfits\': len(outfits),\n                         \'removed_overlap_outfits\': len(original_outfits)-len(outfits),\n                         \'source_overlap_item_ids\': len(overlap), \'usable_images\': count, \'invalid_images\': invalid,\n                         \'positive_triples\': len(positive), \'synthetic_negatives\': len(negatives)}\n        print(split, report[split], flush=True)\n    for a, b in itertools.combinations(seen, 2):\n        if seen[a] & seen[b]:\n            raise ValueError(f\'Item leakage between {a} and {b}\')\n    if not all(examples.values()):\n        raise ValueError(\'At least one split has no complete triple\')\n    dump(out / \'items.json\', items)\n    for split in examples:\n        dump(out / f\'{split}_examples.json\', examples[split])\n        dump(out / f\'{split}_transactions.json\', transactions[split])\n    report[\'protocol\'] = \'3-slot projected positives and synthetic negatives; NOT official benchmark\'\n    report[\'seed\'] = 42\n    report[\'split_policy\'] = \'Keep train; exclude whole held-out outfits sharing any item with earlier original splits. Original files unchanged.\'\n    dump(out / \'preprocessing_report.json\', report)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'run_training_pipeline.py': '"""Sequential local CPU workflow. No UI is created by this training pipeline.\n\nEvery subprocess writes a separate log, and the status file records real exits.\n"""\nimport argparse\nfrom datetime import datetime, timezone\nimport json\nfrom pathlib import Path\nimport subprocess\nimport sys\n\nROOT = Path(__file__).absolute().parents[1]\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--profile\', choices=[\'pilot\', \'full\'], default=\'full\')\n    parser.add_argument(\'--start-at\', help=\'Continue from a named stage; earlier artifacts must already exist\')\n    parser.add_argument(\'--resume\', type=Path, help=\'Checkpoint for the first selected image training stage\')\n    args = parser.parse_args()\n    name = args.profile\n    limited = name == \'pilot\'\n    tasks = [\n        (\'rules\', [\'scripts/train_rules.py\']),\n        (\'metadata_baseline\', [\'scripts/train_compatibility.py\']),\n        (\'metadata_ranking\', [\'scripts/evaluate_recommendation.py\']),\n        (\'embedding\', [\'scripts/train_embedding.py\', \'--run-name\', name, \'--epochs\', \'1\' if limited else \'3\'] +\n                      ([\'--limit\', \'256\', \'--max-steps\', \'8\'] if limited else [])),\n    ]\n    if not limited:\n        tasks.extend([\n            (\'export_embeddings\', [\'scripts/export_embeddings.py\', \'--checkpoint\', \'artifacts/embedding/full/best.pt\']),\n            (\'image_compatibility\', [\'scripts/train_compatibility.py\', \'--embeddings\', \'artifacts/embedding/catalog_vectors.npz\']),\n            (\'image_ranking\', [\'scripts/evaluate_recommendation.py\', \'--embeddings\', \'artifacts/embedding/catalog_vectors.npz\']),\n        ])\n    if args.start_at:\n        names = [name for name, _ in tasks]\n        if args.start_at not in names:\n            parser.error(\'Unknown start stage: \' + args.start_at)\n        tasks = tasks[names.index(args.start_at):]\n    if args.resume:\n        if tasks[0][0] != \'embedding\' or not args.resume.is_file():\n            parser.error(\'Select an image training stage and an existing checkpoint\')\n        tasks[0][1].extend([\'--resume\', str(args.resume.resolve())])\n    (ROOT / \'logs\').mkdir(exist_ok=True)\n    status_path = ROOT / \'reports\' / f\'pipeline_{name}.json\'\n    status_path.parent.mkdir(exist_ok=True)\n    state = {\'profile\': name, \'status\': \'running\', \'steps\': [], \'ui_created\': False}\n    for label, arguments in tasks:\n        state[\'active_step\'] = label\n        state[\'updated_at\'] = datetime.now(timezone.utc).isoformat()\n        status_path.write_text(json.dumps(state, indent=2))\n        print(f\'START {label}\', flush=True)\n        mode = \'a\' if args.resume and label == tasks[0][0] else \'w\'\n        with (ROOT / \'logs\' / f\'{name}_{label}.log\').open(mode, encoding=\'utf-8\') as log:\n            if mode == \'a\':\n                print(f\'Resuming from {args.resume} at {state["updated_at"]}\', file=log, flush=True)\n            result = subprocess.run([sys.executable, \'-X\', \'utf8\', \'-u\', *arguments], cwd=ROOT, stdout=log, stderr=subprocess.STDOUT)\n        state[\'steps\'].append({\'name\': label, \'exit_code\': result.returncode})\n        if result.returncode:\n            state[\'status\'] = \'failed\'; status_path.write_text(json.dumps(state, indent=2))\n            raise SystemExit(f\'Failed {label}; inspect logs/{name}_{label}.log\')\n        print(f\'FINISHED {label}\', flush=True)\n    state[\'status\'] = \'training_finished_requires_result_review\'\n    state[\'active_step\'] = None\n    state[\'updated_at\'] = datetime.now(timezone.utc).isoformat()\n    status_path.write_text(json.dumps(state, indent=2))\n\n\nif __name__ == \'__main__\': main()\n', 'train_compatibility.py': '"""LR/RF experiments with metadata baseline or a versioned image embedding file.\n\nEvaluation uses project\'s synthetic-negative triple protocol, not official AUC.\n"""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport time\nimport joblib\nimport numpy as np\nfrom sklearn.feature_extraction import DictVectorizer\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import roc_auc_score, f1_score, accuracy_score, confusion_matrix\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.pipeline import make_pipeline\n\nROOT = Path(__file__).absolute().parents[1]\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--embeddings\', type=Path)\n    args = parser.parse_args()\n    data = ROOT / \'data/processed/polyvore\'\n    items = json.loads((data / \'items.json\').read_text())\n    rows = {s: json.loads((data / f\'{s}_examples.json\').read_text()) for s in (\'train\', \'validation\', \'test\')}\n    def metadata_features(r):\n        cats = [items[i][\'category\'] for i in r[\'item_ids\']]\n        return {\'top\': cats[0], \'bottom\': cats[1], \'shoes\': cats[2],\n                \'top_bottom\': cats[0] + \'|\' + cats[1], \'top_shoes\': cats[0] + \'|\' + cats[2],\n                \'bottom_shoes\': cats[1] + \'|\' + cats[2]}\n    vectorizer = DictVectorizer(sparse=False)\n    vectorizer.fit([metadata_features(r) for r in rows[\'train\']])\n    features = {s: vectorizer.transform([metadata_features(r) for r in rs]) for s, rs in rows.items()}\n    version = \'metadata_baseline\'\n    if args.embeddings:\n        archive = np.load(args.embeddings, allow_pickle=False)\n        em = dict(zip(archive[\'ids\'].astype(str), archive[\'vectors\']))\n        for split, rs in rows.items():\n            vectors = []\n            for r in rs:\n                a, b, c = [em[i] for i in r[\'item_ids\']]\n                vectors.append(np.concatenate([a, b, c, a*b, a*c, b*c]))\n            features[split] = np.hstack([features[split], vectors])\n        version = \'image_\' + hashlib.sha256(args.embeddings.read_bytes()).hexdigest()[:12]\n    labels = {s: np.array([r[\'label\'] for r in rs]) for s, rs in rows.items()}\n    best = {}\n    validation_results = []\n    for family in (\'logistic_regression\', \'random_forest\'):\n        for parameter in ([0.1, 1.0, 10.0] if family == \'logistic_regression\' else [8, 16, None]):\n            if family == \'logistic_regression\':\n                model = make_pipeline(StandardScaler(), LogisticRegression(C=parameter, max_iter=1500, random_state=42))\n            else:\n                model = RandomForestClassifier(n_estimators=200, max_depth=parameter, min_samples_leaf=3,\n                                               random_state=42, n_jobs=4)\n            start = time.perf_counter()\n            model.fit(features[\'train\'], labels[\'train\'])\n            score = model.predict_proba(features[\'validation\'])[:, 1]\n            auc = float(roc_auc_score(labels[\'validation\'], score))\n            thresholds = np.linspace(0.1, 0.9, 33)\n            threshold = float(max(thresholds, key=lambda t: f1_score(labels[\'validation\'], score >= t)))\n            validation_results.append({\'family\': family, \'parameter\': parameter, \'validation_auc\': auc,\n                                       \'threshold\': threshold, \'seconds\': time.perf_counter() - start})\n            if family not in best or auc > best[family][\'auc\']:\n                best[family] = {\'model\': model, \'auc\': auc, \'threshold\': threshold, \'parameter\': parameter}\n            print(validation_results[-1], flush=True)\n    out = ROOT / \'artifacts/compatibility\' / version\n    out.mkdir(parents=True, exist_ok=True)\n    results = {\'feature_version\': version, \'protocol\': \'project triples with synthetic same-slot negatives\',\n               \'counts\': {s: len(rs) for s, rs in rows.items()}, \'validation_search\': validation_results, \'test\': {}}\n    for family, value in best.items():\n        score = value[\'model\'].predict_proba(features[\'test\'])[:, 1]\n        pred = score >= value[\'threshold\']\n        results[\'test\'][family] = {\'auc\': float(roc_auc_score(labels[\'test\'], score)),\n                                  \'f1\': float(f1_score(labels[\'test\'], pred)),\n                                  \'accuracy\': float(accuracy_score(labels[\'test\'], pred)),\n                                  \'confusion_matrix\': confusion_matrix(labels[\'test\'], pred).tolist()}\n        joblib.dump({\'model\': value[\'model\'], \'vectorizer\': vectorizer, \'threshold\': value[\'threshold\'],\n                     \'feature_version\': version}, out / f\'{family}.joblib\')\n    rp = ROOT / \'reports/metrics\'\n    rp.mkdir(parents=True, exist_ok=True)\n    (rp / f\'compatibility_{version}.json\').write_text(json.dumps(results, indent=2))\n    print(json.dumps(results[\'test\'], indent=2))\n\n\nif __name__ == \'__main__\':\n    main()\n', 'train_embedding.py': '"""ResNet-50 triplet training on the local CPU or CUDA; checkpoint each epoch.\n\nSame-item augmentation positives measure instance retrieval, not human style.\n--limit makes an explicit pilot run. Only projection + layer4 are fine-tuned.\n"""\nimport argparse\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport random\nimport time\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.utils.data import DataLoader, Dataset\nfrom torchvision import models, transforms\nfrom PIL import Image\n\nROOT = Path(__file__).absolute().parents[1]\nos.environ.setdefault(\'TORCH_HOME\', str(ROOT / \'artifacts/torch_cache\'))\n\n\nclass Encoder(nn.Module):\n    def __init__(self, pretrained=True):\n        super().__init__()\n        self.backbone = models.resnet50(weights=models.ResNet50_Weights.DEFAULT if pretrained else None)\n        self.backbone.fc = nn.Linear(self.backbone.fc.in_features, 128)\n        for name, p in self.backbone.named_parameters():\n            p.requires_grad = name.startswith((\'layer4.\', \'fc.\'))\n\n    def forward(self, x):\n        return F.normalize(self.backbone(x), dim=1)\n\n\ndef transform(augment=False):\n    ops = ([transforms.RandomResizedCrop(160, scale=(0.75, 1.0)), transforms.RandomHorizontalFlip()]\n           if augment else [transforms.Resize((160, 160))])\n    return transforms.Compose(ops + [transforms.ToTensor(),\n        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])\n\n\nclass Triplets(Dataset):\n    def __init__(self, items, split, limit=0):\n        self.items = [v for v in items.values() if v[\'split\'] == split]\n        random.Random(42).shuffle(self.items)\n        if limit:\n            self.items = self.items[:limit]\n        self.pools = {s: [r for r in self.items if r[\'slot\'] == s] for s in (\'top\', \'bottom\', \'shoes\')}\n        self.aug = transform(True)\n\n    def __len__(self):\n        return len(self.items)\n\n    def __getitem__(self, idx):\n        row = self.items[idx]\n        candidates = self.pools[row[\'slot\']]\n        if len(candidates) < 2:\n            raise ValueError(\'At least two items per slot required\')\n        negative = random.choice(candidates)\n        while negative[\'item_id\'] == row[\'item_id\']:\n            negative = random.choice(candidates)\n        with Image.open(ROOT / row[\'image_path\']) as im:\n            a, p = self.aug(im.convert(\'RGB\')), self.aug(im.convert(\'RGB\'))\n        with Image.open(ROOT / negative[\'image_path\']) as im:\n            n = self.aug(im.convert(\'RGB\'))\n        return a, p, n\n\n\n@torch.no_grad()\ndef evaluate(model, items, split, device, limit=300):\n    selected = [v for v in items.values() if v[\'split\'] == split]\n    random.Random(123).shuffle(selected)\n    selected = selected[:limit]\n    gallery, queries = [], []\n    model.eval()\n    deterministic = transform()\n    # Fixed mirror query for a reproducible, explicitly limited instance task.\n    for start in range(0, len(selected), 16):\n        images, flipped = [], []\n        for row in selected[start:start+16]:\n            with Image.open(ROOT / row[\'image_path\']) as im:\n                image = im.convert(\'RGB\')\n                images.append(deterministic(image))\n                flipped.append(deterministic(image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)))\n        gallery.append(model(torch.stack(images).to(device)).cpu())\n        queries.append(model(torch.stack(flipped).to(device)).cpu())\n    sims = torch.cat(queries) @ torch.cat(gallery).T\n    for i, r in enumerate(selected):\n        sims[i, torch.tensor([s[\'slot\'] != r[\'slot\'] for s in selected])] = -float(\'inf\')\n    ranks = sims.argsort(dim=1, descending=True)\n    truth = torch.arange(len(selected))[:, None]\n    return {\'recall_at_1\': float((ranks[:, :1] == truth).any(1).float().mean()),\n            \'recall_at_5\': float((ranks[:, :5] == truth).any(1).float().mean()),\n            \'queries\': len(selected), \'protocol\': \'same-item horizontal-flip retrieval, same-slot gallery\'}\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument(\'--epochs\', type=int, default=3)\n    p.add_argument(\'--limit\', type=int, default=0)\n    p.add_argument(\'--batch-size\', type=int, default=8)\n    p.add_argument(\'--max-steps\', type=int, default=0)\n    p.add_argument(\'--margin\', type=float, default=1.0)\n    p.add_argument(\'--resume\', type=Path)\n    p.add_argument(\'--run-name\', default=\'full\')\n    args = p.parse_args()\n    random.seed(42); np.random.seed(42); torch.manual_seed(42); torch.set_num_threads(4)\n    device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    items = json.loads((ROOT / \'data/processed/polyvore/items.json\').read_text())\n    ds = Triplets(items, \'train\', args.limit)\n    loader = DataLoader(ds, batch_size=args.batch_size, shuffle=True, num_workers=0, drop_last=True)\n    model = Encoder(pretrained=not bool(args.resume)).to(device)\n    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)\n    start_epoch, best = 0, -1.0\n    if args.resume:\n        state = torch.load(args.resume, map_location=device, weights_only=False)\n        model.load_state_dict(state[\'model\']); optimizer.load_state_dict(state[\'optimizer\'])\n        start_epoch = state[\'epoch\']; best = state[\'best\']\n    out = ROOT / \'artifacts/embedding\' / args.run_name\n    out.mkdir(parents=True, exist_ok=True)\n    report = {\'device\': str(device), \'train_items\': len(ds), \'pilot\': bool(args.limit or args.max_steps),\n              \'arguments\': {k: str(v) if isinstance(v, Path) else v for k, v in vars(args).items()}, \'epochs\': []}\n    loss_fn = nn.TripletMarginLoss(margin=args.margin)\n    for epoch in range(start_epoch, args.epochs):\n        model.train()\n        # Freeze BN running statistics for reliable small CPU batches.\n        for module in model.modules():\n            if isinstance(module, nn.BatchNorm2d): module.eval()\n        start, total, steps = time.perf_counter(), 0.0, 0\n        for step, (a, pos, neg) in enumerate(loader):\n            optimizer.zero_grad()\n            encoded = model(torch.cat([a, pos, neg]).to(device))\n            loss = loss_fn(*encoded.chunk(3))\n            if not torch.isfinite(loss): raise ValueError(\'Nonfinite loss\')\n            loss.backward(); optimizer.step()\n            total += loss.item(); steps += 1\n            if steps == 1 or steps % 20 == 0:\n                print(f\'epoch={epoch+1} step={steps}/{len(loader)} loss={loss.item():.4f} seconds={time.perf_counter()-start:.1f}\', flush=True)\n            if steps % 100 == 0:\n                torch.save({\'model\': model.state_dict(), \'optimizer\': optimizer.state_dict(),\n                            \'epoch\': epoch, \'best\': best, \'config\': report[\'arguments\'],\n                            \'input_size\': 160, \'embedding_dim\': 128,\n                            \'partial_epoch_steps\': steps, \'resume_policy\': \'restart_current_epoch\'}, out / \'last.pt\')\n            if args.max_steps and steps >= args.max_steps: break\n        if not steps: raise ValueError(\'No training steps\')\n        validation = evaluate(model, items, \'validation\', device)\n        improved = validation[\'recall_at_1\'] > best\n        best = max(best, validation[\'recall_at_1\'])\n        state = {\'model\': model.state_dict(), \'optimizer\': optimizer.state_dict(), \'epoch\': epoch+1,\n                 \'best\': best, \'config\': report[\'arguments\'], \'input_size\': 160, \'embedding_dim\': 128}\n        torch.save(state, out / \'last.pt\')\n        if improved: torch.save(state, out / \'best.pt\')\n        row = {\'epoch\': epoch+1, \'steps\': steps, \'mean_loss\': total/steps, \'seconds\': time.perf_counter()-start, \'validation\': validation}\n        report[\'epochs\'].append(row)\n        (out / \'training_report.json\').write_text(json.dumps(report, indent=2))\n        print(json.dumps(row), flush=True)\n    if not (out / \'best.pt\').exists():\n        raise ValueError(\'No completed epoch checkpoint in this run; resume with a larger epoch target\')\n    model.load_state_dict(torch.load(out / \'best.pt\', map_location=device, weights_only=False)[\'model\'])\n    report[\'test\'] = evaluate(model, items, \'test\', device)\n    (out / \'training_report.json\').write_text(json.dumps(report, indent=2))\n\n\nif __name__ == \'__main__\':\n    main()\n', 'train_rules.py': '"""Train association rules only on train; compare identical Apriori/FP-Growth settings."""\nimport json\nfrom pathlib import Path\nimport time\nimport pandas as pd\nfrom mlxtend.preprocessing import TransactionEncoder\nfrom mlxtend.frequent_patterns import apriori, fpgrowth, association_rules\n\nROOT = Path(__file__).absolute().parents[1]\n\n\ndef main():\n    p = ROOT / \'data/processed/polyvore\'\n    train = json.loads((p / \'train_transactions.json\').read_text())\n    encoder = TransactionEncoder()\n    frame = pd.DataFrame(encoder.fit_transform(train), columns=encoder.columns_)\n    out = ROOT / \'artifacts/mining\'\n    out.mkdir(parents=True, exist_ok=True)\n    report, results = {}, {}\n    for name, fn in [(\'apriori\', apriori), (\'fpgrowth\', fpgrowth)]:\n        start = time.perf_counter()\n        sets = fn(frame, min_support=0.005, use_colnames=True, max_len=3)\n        rules = association_rules(sets, metric=\'confidence\', min_threshold=0.15)\n        rules = rules[(rules[\'lift\'] > 1) & rules[\'consequents\'].map(lambda s: len(s) == 1)]\n        report[name] = {\'seconds\': time.perf_counter() - start, \'itemsets\': len(sets), \'rules\': len(rules)}\n        results[name] = set(sets[\'itemsets\'])\n        serial = [{\'antecedents\': sorted(r[\'antecedents\']), \'consequents\': sorted(r[\'consequents\']),\n                   \'support\': float(r[\'support\']), \'confidence\': float(r[\'confidence\']), \'lift\': float(r[\'lift\'])}\n                  for _, r in rules.iterrows()]\n        (out / f\'{name}_rules.json\').write_text(json.dumps(serial, indent=2), encoding=\'utf-8\')\n        for split in (\'validation\', \'test\'):\n            ts = json.loads((p / f\'{split}_transactions.json\').read_text())\n            report[name][split + \'_antecedent_coverage\'] = sum(any(set(r[\'antecedents\']) <= set(t) for r in serial) for t in ts) / len(ts)\n    assert results[\'apriori\'] == results[\'fpgrowth\']\n    report[\'same_frequent_itemsets\'] = True\n    report[\'train_transactions\'] = len(train)\n    report[\'settings\'] = {\'min_support\': 0.005, \'min_confidence\': 0.15, \'max_len\': 3}\n    rp = ROOT / \'reports/metrics\'\n    rp.mkdir(parents=True, exist_ok=True)\n    (rp / \'rules.json\').write_text(json.dumps(report, indent=2))\n    print(json.dumps(report, indent=2))\n\n\nif __name__ == \'__main__\':\n    main()\n'}
for name, source in sources.items():
    (root / "scripts" / name).write_text(source, encoding="utf-8")
print(*sources, sep="\n")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
saved = Path("/content/drive/MyDrive/PhoiDoTraining") / MODEL / "full"
saved.mkdir(parents=True, exist_ok=True)
out = root / "artifacts" / MODEL / "full"
out.parent.mkdir(parents=True, exist_ok=True)
if not out.exists():
    out.symlink_to(saved, target_is_directory=True)
assert out.resolve() == saved.resolve()
print("Checkpoint và báo cáo:", saved)
print("Checkpoint hiện có:", [p.name for p in saved.glob("*.pt")])

### Tải và tiền xử lý Polyvore
Tài khoản Hugging Face cần quyền truy cập mvasil/polyvore-outfits. Nhập token vào ô đăng nhập riêng khi được yêu cầu.

In [ ]:
dataset = "polyvore"
if dataset == "polyvore":
    from huggingface_hub import get_token, login
    if not get_token():
        from getpass import getpass
        login(token=getpass("Hugging Face read token: "), add_to_git_credential=False)
run("scripts/download_datasets.py", "--dataset", dataset)
run(f"scripts/preprocess_{dataset}.py")
print((root / "data/processed" / dataset / "preprocessing_report.json").read_text())

In [ ]:
report = saved / "training_report.json"
last = saved / "last.pt"
if report.exists() and json.loads(report.read_text()).get("test"):
    print("Đã có báo cáo test. Kiểm tra báo cáo ở ô cuối.")
else:
    if report.exists() and not last.exists():
        raise RuntimeError("Đã có báo cáo nhưng thiếu last.pt. Kiểm tra thư mục Drive trước khi chạy.")
    cmd = [f"scripts/train_{MODEL}.py", "--run-name", "full", "--epochs", "3"]
    if last.is_file():
        cmd += ["--resume", last]
        print("Tiếp tục từ", last)
    else:
        print("Bắt đầu lượt mới", MODEL)
    run(*cmd)

In [ ]:
metrics = json.loads((saved / "training_report.json").read_text())
assert not metrics["pilot"] and "test" in metrics and (saved / "best.pt").is_file(), "Chưa có đủ kết quả cuối."
print(json.dumps(metrics["test"], indent=2))
print("Gửi lại hai tệp:", saved / "best.pt", "và", saved / "training_report.json")